# Indeksy w pandas — przewodnik od podstaw do poziomu eksperckiego

**Czym jest indeks:** to nie jest "jeszcze jedna kolumna" — to oddzielna struktura opisująca ETYKIETY wierszy (i opcjonalnie kolumn), z własnym silnikiem wyszukiwania. Każdy `DataFrame`/`Series` ma indeks zawsze, nawet jeśli go nie ustawiłeś świadomie — wtedy jest to domyślny `RangeIndex` (0, 1, 2, ...).

**Dlaczego to ważne:** indeks robi w pandas dwie rzeczy naraz, które w Excelu czy czystym SQL są rozdzielone albo nie istnieją:
1. **Szybkie wyszukiwanie** — `.loc["North"]` przy odpowiednim indeksie jest rzędy wielkości szybsze niż filtrowanie po kolumnie (patrz Sekcja 3).
2. **Bezpieczne wyrównywanie danych (alignment)** — operacje między dwoma `Series`/`DataFrame` dopasowują wiersze PO ETYKIECIE, nie po pozycji. To chroni przed cichym pomieszaniem danych, gdy kolejność wierszy się różni — ale też potrafi zaskoczyć, jeśli myślisz o danych pozycyjnie, jak w Excelu (patrz Sekcja 7).

**Struktura notatnika:** od podstaw (czym jest, jak ustawiać/resetować), przez selekcję i duplikaty, po alignment, `MultiIndex` i pułapki — w kolejności, w jakiej warto to poznawać.

## Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "region": ["North", "North", "South", "South", "East", "East"],
    "month": ["Jan", "Feb", "Jan", "Feb", "Jan", "Feb"],
    "sales": [1200, 1100, 900, 950, 700, 720],
})
df

## Sekcja 1 — Czym jest indeks

Domyślny indeks to `RangeIndex` — kolejne liczby całkowite, generowane "leniwie" (nie zajmują pamięci jako pełna tablica, dopóki nie jest to konieczne). Indeks ma własny `dtype`, własną nazwę (`name`), i — kluczowe — **nie musi być unikalny** (patrz Sekcja 6).

In [ ]:
print(df.index)
print(type(df.index))
print(f"dtype: {df.index.dtype}, nazwa: {df.index.name}")

## Sekcja 2 — Ustawianie i resetowanie: `set_index` / `reset_index`

- `set_index("kolumna")` — kolumna PRZESTAJE być kolumną, staje się indeksem.
- `set_index(["a", "b"])` — dwie (lub więcej) kolumny naraz tworzą `MultiIndex` (Sekcja 8).
- `set_index(..., append=True)` — DOKŁADA kolejny poziom do istniejącego indeksu, zamiast go zastępować.
- `reset_index()` — odwrotność: indeks wraca jako zwykła kolumna (lub kolumny, jeśli `MultiIndex`).
- `reset_index(drop=True)` — indeks znika **bezpowrotnie**, nie trafia z powrotem jako kolumna — nowy, domyślny `RangeIndex` zajmuje jego miejsce.

In [ ]:
by_region = df.set_index("region")
by_region

In [ ]:
by_region_month = df.set_index(["region", "month"])
by_region_month

In [ ]:
# append=True - dokładamy 'month' jako DRUGI poziom do istniejącego indeksu 'region'
by_region.set_index("month", append=True)

In [ ]:
print(by_region_month.reset_index())  # indeks wraca jako kolumny
print()
print(by_region_month.reset_index(drop=True))  # indeks znika bezpowrotnie

## Sekcja 3 — Dlaczego warto pracować na indeksach: wydajność

`.loc` na dobrze ustawionym indeksie używa wewnętrznego **hash-engine'u** do wyszukiwania — podobnie jak indeks w bazie danych. Ale ten silnik trzeba najpierw ZBUDOWAĆ — pierwsze użycie `.loc` na świeżo ustawionym indeksie kosztuje jednorazowo więcej niż samo filtrowanie. Dopiero KOLEJNE odwołania są dramatycznie szybsze.

In [ ]:
import time

rng = np.random.default_rng(4)
n = 500_000
codes = [f"C{i:06d}" for i in range(n)]
big = pd.DataFrame({"code": codes, "value": rng.integers(1, 1000, n)})
target = codes[n // 2]

big_indexed = big.set_index("code")

start = time.perf_counter()
_ = big_indexed.loc[target]  # PIERWSZE użycie - buduje hash-engine
t_first = time.perf_counter() - start

start = time.perf_counter()
_ = big[big["code"] == target]  # filtrowanie po kolumnie
t_filter = time.perf_counter() - start

start = time.perf_counter()
_ = big_indexed.loc[target]  # KOLEJNE użycie - silnik już zbudowany
t_second = time.perf_counter() - start

print(f"Pierwsze .loc (budowa silnika): {t_first*1000:.2f} ms")
print(f"Filtrowanie po kolumnie:        {t_filter*1000:.2f} ms")
print(f"Kolejne .loc (silnik gotowy):   {t_second*1000:.2f} ms  <- {t_filter/t_second:.0f}x szybciej niż filtrowanie")

**Wniosek praktyczny:** jeśli robisz JEDNO wyszukiwanie, `set_index()` się nie opłaca — koszt budowy silnika przewyższa oszczędność. Jeśli robisz wiele wyszukiwań na tych samych danych (np. w pętli albo w wielu miejscach notebooka), `set_index()` raz na początku i korzystanie z `.loc` wielokrotnie jest zdecydowanie szybsze niż filtrowanie za każdym razem od nowa.

## Sekcja 4 — Aktualizacja indeksu

- `rename_axis("nazwa")` — nadaje/zmienia NAZWĘ indeksu (nie zmienia etykiet).
- `rename(index={...})` — podmienia KONKRETNE etykiety, resztę zostawia bez zmian.
- Bezpośrednie przypisanie `.index = [...]` — nadpisuje CAŁY indeks nową listą (musi mieć dokładnie tyle samo elementów co wierszy).

In [ ]:
s = pd.Series([300, 100, 700, 200], index=["North", "South", "East", "West"])

print(s.rename_axis("region"))
print()
print(s.rename(index={"North": "Polnoc"}))  # tylko jedna etykieta zmieniona

## Sekcja 5 — Wybieranie i filtrowanie po indeksie

`.loc[]` przyjmuje: pojedynczą etykietę, listę etykiet, albo slice `"a":"b"` (WŁĄCZNIE z obiema końcówkami — inaczej niż przy pozycyjnym `.iloc`!). `.index.isin()` przydaje się do filtrowania warunkowego, gdy potrzebna logika boolowska na samych etykietach.

In [ ]:
print(by_region.loc["North"])        # pojedyncza etykieta
print()
print(by_region.loc[["North", "East"]])  # lista etykiet
print()
print(by_region.loc[by_region.index.isin(["North", "South"])])  # filtr po etykietach

## Sekcja 6 — Weryfikacja duplikatów w indeksie

Indeks **nie musi być unikalny** — pandas na to pozwala, ale konsekwencje przy `.loc` są poważne: zamiast pojedynczej wartości/wiersza, dostajesz `DataFrame` ze WSZYSTKIMI dopasowaniami. Kod pisany z założeniem "jeden wiersz na etykietę" wybuchnie (albo, gorzej, cicho zwróci błędny kształt danych) na indeksie z duplikatem.

In [ ]:
dup = pd.DataFrame({"sales": [100, 200, 300]}, index=["A", "B", "A"])

print(f"Indeks unikalny? {dup.index.is_unique}")
print(f"Które etykiety są duplikatem: {dup.index.duplicated().tolist()}")
print()

print("dup.loc['A'] - zwraca DataFrame, nie pojedynczy wiersz:")
print(dup.loc["A"])
print(f"Typ: {type(dup.loc['A']).__name__}")
print()

unique = pd.DataFrame({"sales": [100, 200, 300]}, index=["A", "B", "C"])
print("Dla porównania, na unikalnym indeksie .loc['A'] zwraca Series:")
print(unique.loc["A"])
print(f"Typ: {type(unique.loc['A']).__name__}")

## Sekcja 7 — Arytmetyka i wyrównywanie (alignment)

To najważniejsza koncepcja w tej notatce. Operacje między dwoma `Series`/`DataFrame` **dopasowują wiersze po etykiecie indeksu, nie po pozycji**. Dla kogoś przyzwyczajonego do Excela (gdzie komórka `A2 + B2` zawsze oznacza "ten sam wiersz") to bywa mylące — ale to właśnie ta mechanika chroni przed cichym pomieszaniem danych.

### Strona A: wygląda jak pułapka — różne zestawy etykiet dają `NaN`

In [ ]:
s1 = pd.Series([100, 200, 300], index=["North", "South", "East"])
s2 = pd.Series([10, 20, 30], index=["South", "East", "West"])

print(s1)
print()
print(s2)
print()
print("s1 + s2 - dopasowanie PO ETYKIECIE, nie po pozycji:")
print(s1 + s2)
print()
print("'North' i 'West' dostają NaN - bo nie miały pary w drugiej serii.")
print("Ktoś myślący pozycyjnie (jak w Excelu) oczekiwałby zupełnie innych wyników.")

### Strona B: ten sam mechanizm ratuje przed cichym błędem

Gdyby pandas dopasowywał POZYCYJNIE (jak Excel), różna kolejność wierszy w dwóch tabelach dałaby błędny, ale "cichy" wynik — bez żadnego ostrzeżenia. Dzięki alignment po etykiecie, kolejność wierszy nie ma znaczenia.

In [ ]:
df1 = pd.DataFrame({"sales": [100, 200, 300]}, index=["North", "South", "East"])
df2 = pd.DataFrame({"sales": [999, 111, 222]}, index=["East", "North", "South"])  # INNA kolejność!

print(df1)
print()
print(df2)
print()
print("df1 + df2 - mimo innej kolejności wierszy, wynik jest MERYTORYCZNIE poprawny:")
print(df1 + df2)
print()
print("North: 100+111=211 (poprawnie dopasowane, mimo że df2 ma North na 2. pozycji, nie 1.)")

## Sekcja 8 — `MultiIndex`: czym jest

Hierarchiczny indeks — kilka poziomów etykiet naraz (np. `region` + `month`). Powstaje naturalnie z `set_index()` na kilku kolumnach (Sekcja 2) albo z wyniku `groupby()` po kilku kluczach.

In [ ]:
multi = df.set_index(["region", "month"])
print(multi)
print()
print(f"Poziomy: {multi.index.names}")
print(f"Liczba poziomów: {multi.index.nlevels}")

## Sekcja 9 — Selekcja na `MultiIndex`

Cztery sposoby, od najprostszego do najbardziej elastycznego:
1. `.loc[("North", "Jan")]` — konkretna kombinacja wszystkich poziomów, jako krotka.
2. `.loc["North"]` — częściowe indeksowanie: podajesz tylko PIERWSZY poziom, dostajesz wszystkie pasujące wiersze z pozostałymi poziomami jako nowy indeks.
3. `.xs("Jan", level="month")` — cross-section: wybór po WEWNĘTRZNYM poziomie, bez podawania pierwszego.
4. `pd.IndexSlice` (zwyczajowo importowane jako `idx`) — najbardziej elastyczne: złożone wycinanie po WIELU poziomach jednocześnie, z możliwością pominięcia poziomu przez `slice(None)`.

In [ ]:
print(multi.loc[("North", "Jan")])

In [ ]:
multi.loc["North"]

In [ ]:
multi.xs("Jan", level="month")

In [ ]:
idx = pd.IndexSlice

# Kilka regionów naraz, konkretny miesiąc
print(multi.loc[idx[["North", "South"], "Jan"], :])
print()

# slice(None) - pomiń poziom 'region', weź WSZYSTKIE regiony, tylko miesiąc 'Feb'
print(multi.loc[idx[:, "Feb"], :])

## Sekcja 10 — `idxmax` / `idxmin`: etykieta, nie wartość

Najczęstsza pomyłka: `idxmax()` zwraca ETYKIETĘ indeksu, przy której wartość jest maksymalna — nie samą wartość. Żeby dostać jedno i drugie naraz, potrzebne są dwa wywołania (albo `.loc[s.idxmax()]`, co jest tym samym co `s.max()` dla prostej Series, ale przydaje się przy `DataFrame`).

In [ ]:
s = pd.Series([300, 100, 700, 200], index=["North", "South", "East", "West"])

print(f"Wartość maksymalna: {s.max()}")
print(f"idxmax() - ETYKIETA regionu z max wartością: {s.idxmax()}")
print(f"idxmin() - ETYKIETA regionu z min wartością: {s.idxmin()}")

## Sekcja 11 — Indeks w `groupby()` i `pivot()`

Szczegóły `groupby`/`pivot` mają osobne, głębsze notatki w tym repo — tu tylko to, co dotyczy bezpośrednio indeksu:

- `groupby("region")` **domyślnie** ustawia klucz grupowania jako indeks wyniku (`as_index=True`) — dlatego wynik `groupby().sum()` wygląda inaczej niż zwykły `DataFrame`.
- `as_index=False` zwraca `region` jako zwykłą kolumnę zamiast indeksu.
- `pivot_table(index="region", columns="month")` używa parametru `index=` dosłownie do zbudowania indeksu wyniku.
- `unstack()`/`stack()` (notatka o reshapingu) operują wprost na POZIOMACH `MultiIndex`.

In [ ]:
grouped = df.groupby("region")["sales"].sum()
print(f"Typ indeksu wyniku groupby: {type(grouped.index).__name__}")
print(grouped)
print()
print("as_index=False - 'region' zostaje zwykłą kolumną:")
print(df.groupby("region", as_index=False)["sales"].sum())

## Sekcja 12 — Pułapki

### Pułapka 1 — slice po NIEPOSORTOWANYM indeksie może dać CICHY, PUSTY wynik

To najbardziej niebezpieczna pułapka w tej notatce, bo nie rzuca błędu — po prostu zwraca puste dane, jakby faktycznie nic nie pasowało. `.loc["A":"C"]` na nieposortowanym indeksie nie ma gwarancji poprawnego działania: pandas nie wie, gdzie "zaczyna się" i "kończy" zakres bez posortowanej struktury.

In [ ]:
unsorted = pd.Series([1, 2, 3, 4, 5], index=["C", "A", "D", "B", "E"])
print(unsorted)
print()

print(f"Czy indeks jest posortowany? {unsorted.index.is_monotonic_increasing}")
print()

result = unsorted.loc["A":"C"]
print(f"unsorted.loc['A':'C'] -> {len(result)} wierszy (PUSTO, bez błędu!):")
print(result)
print()

print("Po sort_index() działa poprawnie:")
print(unsorted.sort_index().loc["A":"C"])

**Zabezpieczenie:** sprawdź `.index.is_monotonic_increasing` przed jakimkolwiek slice'owaniem po etykiecie na niepewnych danych, albo po prostu wywołaj `.sort_index()` zawczasu, jeśli planujesz operacje na zakresach.

## Podsumowanie

| Zadanie | Rozwiązanie |
|---|---|
| Kolumna → indeks | `.set_index("col")` |
| Kilka kolumn → `MultiIndex` | `.set_index(["a", "b"])` |
| Dołożenie poziomu do istniejącego indeksu | `.set_index("col", append=True)` |
| Indeks → z powrotem kolumna(y) | `.reset_index()` |
| Usunięcie indeksu bezpowrotnie | `.reset_index(drop=True)` |
| Zmiana nazwy indeksu | `.rename_axis("nazwa")` |
| Podmiana konkretnych etykiet | `.rename(index={...})` |
| Wybór po etykiecie / liście / zakresie | `.loc[etykieta]` / `.loc[[lista]]` / `.loc["a":"b"]` |
| Filtr po zbiorze etykiet | `.index.isin([...])` |
| Czy indeks unikalny | `.index.is_unique` |
| Które etykiety są duplikatem | `.index.duplicated()` |
| Czy indeks posortowany (bezpieczny do slice'owania) | `.index.is_monotonic_increasing` |
| Konkretna kombinacja poziomów `MultiIndex` | `.loc[("a", "b")]` |
| Częściowe indeksowanie (tylko pierwszy poziom) | `.loc["a"]` |
| Wybór po WEWNĘTRZNYM poziomie `MultiIndex` | `.xs("b", level="nazwa_poziomu")` |
| Złożone wycinanie po wielu poziomach naraz | `idx = pd.IndexSlice; df.loc[idx[...], :]` |
| Etykieta wiersza z wartością max/min | `.idxmax()` / `.idxmin()` (NIE sama wartość) |
| Klucz grupowania jako zwykła kolumna, nie indeks | `.groupby("col", as_index=False)` |

**Wniosek:** indeks to fundament tego, jak pandas "myśli" o danych — nie kosmetyczna etykieta, tylko mechanizm odpowiedzialny jednocześnie za wydajność i za bezpieczeństwo operacji między tabelami. Najbardziej podstępna pułapka (Sekcja 12) — jak zresztą wszystkie pułapki w tym repo — nie rzuca błędu. Nawyk `.sort_index()` i `.index.is_unique` przed operacjami na zakresach/duplikatach to najtańsza ochrona.